In [ ]:
import pandas as pd
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import shapely

In [ ]:
# Cleaned version
# full_df = pd.read_csv('./out/sentinel_2021_v7_aea_bordermerged.csv')
# Erosion and dilation first:
for csv in ['./out_erode_dilate/sentinel_2021_v7_aea_dilate_bordermerged.csv',
            './out_erode_dilate/sentinel_2021_v7_aea_erode_bordermerged.csv',
            './out/sentinel_2021_v7_aea_bordermerged.csv']:
    full_df = pd.read_csv(csv)
    gdf = gpd.GeoDataFrame(
        full_df, geometry=gpd.points_from_xy(full_df.center_lon, full_df.center_lat),
        crs='ESRI:102033'
        )
    brazil_gdf = gpd.read_file('../../../../../analysis/data/misc/general_borders/Brazil_aea.shp')
    full_df = gpd.clip(gdf, brazil_gdf)
    full_df['area_ha'] = full_df['area']*100/10000 # HA
    full_df['area_km'] = full_df['area']*100/(1000*1000) # km2
    full_df = full_df.loc[full_df['hydropoly_max']<=50] # Remove less than 100 ha
    full_df = full_df.loc[full_df['area_ha']<=50] # Remove less than 100 ha
    full_df = full_df.loc[full_df['area_ha']>=0.05] 
    full_df = full_df.drop(columns=['border_vals', 'on_border']).rename(columns={'hydropoly_max':'hp_max'})
    # full_df.drop(columns=['geometry']).to_csv('./out/sentinel_2021_v7_aea_cleaned.csv', index=False)

    gdf_wgs84 = full_df.to_crs('EPSG:4326')
    gdf_wgs84['longitude'] = gdf_wgs84.geometry.x
    gdf_wgs84['latitude'] = gdf_wgs84.geometry.y
    gdf_wgs84.drop(columns=['geometry']).to_csv(csv.replace('bordermerged.csv','cleaned.csv'), index=False)
    print(full_df['area_km'].sum())
    print(full_df['area_km'].shape)

In [ ]:
all_csvs = glob.glob('./out/sentinel_2021_v7_aea_cleaned.csv')
all_csvs.sort()

In [ ]:
def read_process_csv(csv):
    temp_df = pd.read_csv(csv)
    temp_df['satellite'] = os.path.basename(csv)[:8]
    temp_df['year'] = int(os.path.basename(csv)[9:13])
    return temp_df

In [ ]:
full_df = pd.read_csv('./out/sentinel_2021_v7_wgs84_cleaned.csv')

In [ ]:
lulc_df = pd.read_csv('../lulc/out/lulc_stats_res_sentinel_v7_2021_summarized.csv')

# Basic area distribution stats

In [ ]:
print(full_df.loc[(full_df.area_ha > 10), 'area_km'].count())
print(full_df.loc[(full_df.area_ha > 10), 'area_km'].count()/full_df.shape[0])
print(full_df.loc[(full_df.area_ha < 1), 'area_km'].count()/full_df.shape[0])

In [ ]:
total_area = full_df['area_km'].sum()
print(full_df['area_km'].sum())
print(full_df['area_ha'].sum())
print(full_df.loc[(full_df.area_ha > 10), 'area_km'].sum())
print(full_df.loc[(full_df.area_ha > 10), 'area_km'].sum()/total_area)

print(full_df.loc[(full_df.area_ha < 1), 'area_km'].sum())
print(full_df.loc[(full_df.area_ha < 1), 'area_km'].sum()/total_area)

In [ ]:
print(full_df['area_ha'].median())
print(full_df['area_ha'].mean())

In [ ]:
plt.hist(full_df['area_ha'], bins=np.arange(0, 5.01, 0.5))